# Knowledge Graph: entity & relationship extraction

Flow: **load chunks → schema → prompt → extract one chunk → normalize names → validate → test on 15 chunks → review**

What changed after the first test (5 AGI chunks, only ~3 of 19 relations were correct):

| Problem in v1 | Fix here |
|---|---|
| `IS_A` / `RELATED_TO` / `INTRODUCES` used for everything ("GPT-4 IS_A mathematics") | Removed the vague types. Added `HAS_CAPABILITY`, `OUTPERFORMS`, `DEVELOPED_BY`, `EVALUATED_ON`… each with a definition and direction |
| Fields typed as DATASET, authors typed as ORGANIZATION | New `DOMAIN` type. Front matter, acknowledgments and author lists are skipped |
| "Artificial intelligence" vs "artificial intelligence", LLM vs large language model | `aliases` field + one alias map + case-insensitive names |
| Correct relations rejected because of `GPT--4` or curly quotes | Quotes and dashes are normalized before matching |
| Wrong relations accepted because the quote existed somewhere | **Both entity names must appear inside the quote**, type rules (e.g. `DEVELOPED_BY` must point to an ORGANIZATION), one quote can't back more than 3 relations |
| Test only covered 5 AGI chunks | Test set = 3 chunks from **each** of the 5 papers |

**v3 changes** (after the v2 test: 6 of 14 relations correct, 43%):

| Problem in v2 | Fix in v3 |
|---|---|
| Citations became entities ("Mei et al.", "Devlin et al. 2019") | Entities with `et al.` or `[...]` are dropped, and the prompt says so |
| Good relations rejected: the quote skipped "GPT-4" or dropped `[362]` | Citation markers removed before matching; names are checked in the **whole sentence** around the quote |
| "max 3 relations per quote" killed correct ones (GPT-4 IS_A LLM) | Rule removed. Instead: only one relation per entity pair per chunk |
| HELM "DEVELOPED_BY" language models, misuse risks called limitations | New `EVALUATES` and `HAS_RISK` types; `PART_OF` tightened; ORGANIZATION only allowed in `DEVELOPED_BY` |
| "examples include BERT, GPT-3, CLIP" gave nothing | New prompt example for that pattern |

**v3 test: 12 of 17 correct (71%), 16 of 17 with partials (94%).** After it, two last fixes:
the sentence check no longer runs into the next sentence, and a new `COVERS` relation (MMLU COVERS mathematics).

## 1. Setup and load chunks

In [1]:
import json
import re
import time
from pathlib import Path
from collections import Counter
from typing import Literal

import requests
import pandas as pd
from pydantic import BaseModel, Field

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
CHUNKS_PATH = PROJECT_ROOT / "data" / "processed" / "chunks.jsonl"
ARTIFACTS_DIR = PROJECT_ROOT / "artifacts"
ARTIFACTS_DIR.mkdir(parents=True, exist_ok=True)

OLLAMA_URL = "http://localhost:11434/api/chat"
MODEL_NAME = "qwen3:4b-instruct"

with CHUNKS_PATH.open(encoding="utf-8") as f:
    chunks = [json.loads(line) for line in f if line.strip()]
chunk_by_id = {c["chunk_id"]: c for c in chunks}

In [2]:
# check: chunks loaded
print("chunks:", len(chunks))
print(Counter(c["document"] for c in chunks))
print("fields:", list(chunks[0].keys()))

chunks: 2927
Counter({'LLMs': 754, 'Foundation_Models': 693, 'Holistic_Eval': 605, 'AGI': 562, 'GPT-4': 313})
fields: ['chunk_id', 'document', 'title', 'text', 'section_path', 'page_start', 'page_end']


In [3]:
# check: Ollama is running and the model is installed
tags = requests.get("http://localhost:11434/api/tags", timeout=10).json()
installed = [m["name"] for m in tags.get("models", [])]
print("installed:", installed)
assert MODEL_NAME in installed, f"{MODEL_NAME} not installed. Run: ollama pull {MODEL_NAME}"

installed: ['qwen3:4b-instruct-2507-q4_K_M', 'qwen2.5-0.5b-instruct-fc:latest', 'qwen2.5:0.5b-instruct', 'llama3.2:3b', 'qwen3:4b-instruct']


## 2. Schema: what the model is allowed to return

**Entity types**: what kind of thing a name is.
**Relation types**: each has a fixed **direction** (`source → target`) and a one-line meaning. The meaning goes into the prompt.
There's deliberately **no "RELATED_TO"**: if none of these fits, the model should return nothing.

In [4]:
ENTITY_TYPES = {
    "MODEL":        "a specific AI model or system, e.g. GPT-4, PaLM, BERT",
    "ORGANIZATION": "a company, lab or university, e.g. OpenAI, Google, Stanford",
    "BENCHMARK":    "an evaluation suite or test, e.g. HELM, MMLU, HumanEval",
    "DATASET":      "a collection of data used for training, e.g. Common Crawl, The Pile",
    "METHOD":       "a technique or algorithm, e.g. RLHF, chain-of-thought prompting, transformer",
    "METRIC":       "something measured, e.g. accuracy, calibration, toxicity",
    "TASK":         "a concrete task, e.g. question answering, summarization",
    "DOMAIN":       "a field or subject area, e.g. medicine, law, mathematics, coding",
    "CONCEPT":      "an abstract idea, e.g. artificial general intelligence, homogenization, hallucination",
}

RELATION_TYPES = {
    "DEVELOPED_BY":   "MODEL/BENCHMARK -> ORGANIZATION that built it",
    "IS_A":           "specific thing -> more general category (GPT-4 IS_A large language model)",
    "OUTPERFORMS":    "MODEL/METHOD -> MODEL/METHOD it is explicitly said to beat",
    "EVALUATES":      "BENCHMARK -> MODEL it tests (HELM EVALUATES GPT-3)",
    "COMPARED_WITH":  "A -> B when the text compares them without a clear winner",
    "EVALUATED_ON":   "MODEL -> BENCHMARK/DATASET/TASK it was tested on",
    "MEASURES":       "BENCHMARK/METRIC -> METRIC/CONCEPT it measures (HELM MEASURES calibration)",
    "COVERS":         "BENCHMARK/DATASET -> DOMAIN/TASK it spans (MMLU COVERS mathematics)",
    "TRAINED_ON":     "MODEL -> DATASET it was trained on",
    "USES":           "MODEL/METHOD -> METHOD it uses (GPT-4 USES RLHF)",
    "BUILDS_ON":      "newer thing -> older thing it extends",
    "PART_OF":        "component -> the whole it belongs to (a dataset in a benchmark). NOT for two things listed together",
    "HAS_CAPABILITY": "MODEL -> TASK/DOMAIN it is said to be able to do",
    "HAS_LIMITATION": "MODEL/METHOD -> CONCEPT describing its own weakness (GPT-4 HAS_LIMITATION hallucination)",
    "HAS_RISK":       "MODEL/METHOD -> CONCEPT it could be misused for or cause (GPT-4 HAS_RISK disinformation)",
}

# simple sanity rules: (relation, which end) -> allowed entity types
TYPE_RULES = {
    ("DEVELOPED_BY", "target"):   {"ORGANIZATION"},
    ("TRAINED_ON", "target"):     {"DATASET"},
    ("EVALUATED_ON", "target"):   {"BENCHMARK", "DATASET", "TASK"},
    ("HAS_CAPABILITY", "source"): {"MODEL", "METHOD"},
    ("HAS_CAPABILITY", "target"): {"TASK", "DOMAIN"},
    ("OUTPERFORMS", "source"):    {"MODEL", "METHOD"},
    ("EVALUATES", "source"):      {"BENCHMARK"},
    ("EVALUATES", "target"):      {"MODEL"},
    ("COVERS", "source"):         {"BENCHMARK", "DATASET"},
    ("COVERS", "target"):         {"DOMAIN", "TASK"},
    ("HAS_LIMITATION", "source"): {"MODEL", "METHOD"},
    ("HAS_RISK", "source"):       {"MODEL", "METHOD"},
}

In [5]:
class Entity(BaseModel):
    name: str = Field(description="Full name, singular, as written in the passage (no acronym if the full name is given)")
    entity_type: Literal["MODEL", "ORGANIZATION", "BENCHMARK", "DATASET", "METHOD",
                         "METRIC", "TASK", "DOMAIN", "CONCEPT"]
    aliases: list[str] = Field(default_factory=list, description="Other names used in the passage, e.g. acronyms like LLM, AGI")


class Relationship(BaseModel):
    source: str = Field(description="Entity name (must be one of the extracted entities)")
    relation: Literal["DEVELOPED_BY", "IS_A", "OUTPERFORMS", "EVALUATES", "COMPARED_WITH", "EVALUATED_ON",
                      "MEASURES", "COVERS", "TRAINED_ON", "USES", "BUILDS_ON", "PART_OF",
                      "HAS_CAPABILITY", "HAS_LIMITATION", "HAS_RISK"]
    target: str = Field(description="Entity name (must be one of the extracted entities)")
    evidence: str = Field(description="ONE sentence or clause copied exactly from the passage that states this relation")


class ChunkExtraction(BaseModel):
    entities: list[Entity]
    relationships: list[Relationship]

In [6]:
from typing import get_args

# check: schema matches the dictionaries above
assert set(get_args(Entity.model_fields["entity_type"].annotation or ())) == set(ENTITY_TYPES)
assert set(get_args(Relationship.model_fields["relation"].annotation or ())) == set(RELATION_TYPES)
display(pd.DataFrame([{"relation": k, "meaning": v} for k, v in RELATION_TYPES.items()]))

,relation,meaning
0,DEVELOPED_BY,MODEL/BENCHMARK -> ORGANIZATION that built it
1,IS_A,specific thing -> more general category (GPT-4...
2,OUTPERFORMS,MODEL/METHOD -> MODEL/METHOD it is explicitly ...
3,EVALUATES,BENCHMARK -> MODEL it tests (HELM EVALUATES GP...
4,COMPARED_WITH,A -> B when the text compares them without a c...
5,EVALUATED_ON,MODEL -> BENCHMARK/DATASET/TASK it was tested on
6,MEASURES,BENCHMARK/METRIC -> METRIC/CONCEPT it measures...
7,COVERS,BENCHMARK/DATASET -> DOMAIN/TASK it spans (MML...
8,TRAINED_ON,MODEL -> DATASET it was trained on
9,USES,MODEL/METHOD -> METHOD it uses (GPT-4 USES RLHF)


## 3. Prompt with examples

Small models follow **examples** much better than rules, so the prompt shows four:
1. a passage with good relations,
2. a passage where the right answer is a **list of capabilities** (not `IS_A`),
3. "examples include X, Y" → `IS_A`, a benchmark that `EVALUATES` a model, a misuse risk, and two traps: citations and "X and Y" lists,
4. a passage with **no** relations at all.

In [7]:
EXAMPLES = """
EXAMPLE 1
Passage: The latest model developed by OpenAI, GPT-4, was trained using an unprecedented scale of compute and data. GPT-4 is part of a new cohort of LLMs, along with ChatGPT, and often vastly surpasses prior models such as ChatGPT.
Output:
{"entities": [
  {"name": "GPT-4", "entity_type": "MODEL", "aliases": []},
  {"name": "OpenAI", "entity_type": "ORGANIZATION", "aliases": []},
  {"name": "large language model", "entity_type": "CONCEPT", "aliases": ["LLM"]},
  {"name": "ChatGPT", "entity_type": "MODEL", "aliases": []}],
 "relationships": [
  {"source": "GPT-4", "relation": "DEVELOPED_BY", "target": "OpenAI", "evidence": "The latest model developed by OpenAI, GPT-4"},
  {"source": "GPT-4", "relation": "IS_A", "target": "large language model", "evidence": "GPT-4 is part of a new cohort of LLMs"},
  {"source": "GPT-4", "relation": "OUTPERFORMS", "target": "ChatGPT", "evidence": "often vastly surpasses prior models such as ChatGPT"}]}

EXAMPLE 2
Passage: GPT-4 can solve novel and difficult tasks that span mathematics, coding and medicine. HELM measures calibration and robustness for every model it evaluates.
Output:
{"entities": [
  {"name": "GPT-4", "entity_type": "MODEL", "aliases": []},
  {"name": "mathematics", "entity_type": "DOMAIN", "aliases": []},
  {"name": "coding", "entity_type": "DOMAIN", "aliases": []},
  {"name": "medicine", "entity_type": "DOMAIN", "aliases": []},
  {"name": "Holistic Evaluation of Language Models", "entity_type": "BENCHMARK", "aliases": ["HELM"]},
  {"name": "calibration", "entity_type": "METRIC", "aliases": []},
  {"name": "robustness", "entity_type": "METRIC", "aliases": []}],
 "relationships": [
  {"source": "GPT-4", "relation": "HAS_CAPABILITY", "target": "mathematics", "evidence": "GPT-4 can solve novel and difficult tasks that span mathematics"},
  {"source": "Holistic Evaluation of Language Models", "relation": "MEASURES", "target": "calibration", "evidence": "HELM measures calibration"},
  {"source": "Holistic Evaluation of Language Models", "relation": "MEASURES", "target": "robustness", "evidence": "HELM measures calibration and robustness"}]}

EXAMPLE 3
Passage: A foundation model is any model trained on broad data; current examples include BERT [Devlin et al. 2019] and GPT-3 [Brown et al. 2020]. HELM evaluates 30 language models, including GPT-3. Models like GPT-4 can be misused to generate disinformation. Healthcare and biomedicine are large application areas.
Output:
{"entities": [
  {"name": "foundation model", "entity_type": "CONCEPT", "aliases": []},
  {"name": "BERT", "entity_type": "MODEL", "aliases": []},
  {"name": "GPT-3", "entity_type": "MODEL", "aliases": []},
  {"name": "Holistic Evaluation of Language Models", "entity_type": "BENCHMARK", "aliases": ["HELM"]},
  {"name": "GPT-4", "entity_type": "MODEL", "aliases": []},
  {"name": "disinformation", "entity_type": "CONCEPT", "aliases": []},
  {"name": "healthcare", "entity_type": "DOMAIN", "aliases": []},
  {"name": "biomedicine", "entity_type": "DOMAIN", "aliases": []}],
 "relationships": [
  {"source": "BERT", "relation": "IS_A", "target": "foundation model", "evidence": "A foundation model is any model trained on broad data; current examples include BERT"},
  {"source": "GPT-3", "relation": "IS_A", "target": "foundation model", "evidence": "A foundation model is any model trained on broad data; current examples include BERT [Devlin et al. 2019] and GPT-3"},
  {"source": "Holistic Evaluation of Language Models", "relation": "EVALUATES", "target": "GPT-3", "evidence": "HELM evaluates 30 language models, including GPT-3"},
  {"source": "GPT-4", "relation": "HAS_RISK", "target": "disinformation", "evidence": "Models like GPT-4 can be misused to generate disinformation"}]}
(Note: no entity for "Devlin et al." or "Brown et al." - they are citations. No relation between healthcare and biomedicine - being listed together is not a relation.)

EXAMPLE 4
Passage: Intelligence is a multifaceted and elusive concept that has long challenged psychologists, philosophers, and computer scientists.
Output:
{"entities": [], "relationships": []}
"""

SYSTEM_PROMPT = f"""You extract a knowledge graph from passages of AI/ML research papers.

ENTITY TYPES:
{chr(10).join(f"- {k}: {v}" for k, v in ENTITY_TYPES.items())}

RELATION TYPES (direction is source -> target):
{chr(10).join(f"- {k}: {v}" for k, v in RELATION_TYPES.items())}

RULES:
1. Use ONLY the passage. No outside knowledge.
2. Extract specific, meaningful entities. NEVER make an entity from a person, an author or a citation
   ("Devlin et al.", "Mei et al. (2021)", "[Ope23]"). Skip generic words (system, paper, results, tasks).
3. Entity names: full name, singular, as written. Put acronyms in aliases (LLM, AGI, HELM).
4. Only add a relation when the passage STATES it. Two names in the same sentence is NOT enough.
5. If no relation type fits exactly, leave the relation out. Empty lists are fine.
   PART_OF is only for a component inside a larger whole. Things listed together ("X and Y") are NOT related.
   ORGANIZATION entities are only used with DEVELOPED_BY.
6. evidence = one short sentence or clause copied exactly from the passage, containing BOTH entity names (or their aliases).
7. source and target must be names from your entities list.

{EXAMPLES}"""

print(SYSTEM_PROMPT[:1500], "...")

You extract a knowledge graph from passages of AI/ML research papers.

ENTITY TYPES:
- MODEL: a specific AI model or system, e.g. GPT-4, PaLM, BERT
- ORGANIZATION: a company, lab or university, e.g. OpenAI, Google, Stanford
- BENCHMARK: an evaluation suite or test, e.g. HELM, MMLU, HumanEval
- DATASET: a collection of data used for training, e.g. Common Crawl, The Pile
- METHOD: a technique or algorithm, e.g. RLHF, chain-of-thought prompting, transformer
- METRIC: something measured, e.g. accuracy, calibration, toxicity
- TASK: a concrete task, e.g. question answering, summarization
- DOMAIN: a field or subject area, e.g. medicine, law, mathematics, coding
- CONCEPT: an abstract idea, e.g. artificial general intelligence, homogenization, hallucination

RELATION TYPES (direction is source -> target):
- DEVELOPED_BY: MODEL/BENCHMARK -> ORGANIZATION that built it
- IS_A: specific thing -> more general category (GPT-4 IS_A large language model)
- OUTPERFORMS: MODEL/METHOD -> MODEL/METHOD i

## 4. Extract one chunk

`format=` passes the JSON schema to Ollama, so the answer always has the right shape. `temperature=0` makes it repeatable.

In [8]:
def extract_chunk(chunk):
    user_prompt = f"""Paper: {chunk["title"]}
Section: {" > ".join(chunk["section_path"])}

Passage: {chunk["text"]}
Output:"""

    response = requests.post(
        OLLAMA_URL,
        json={
            "model": MODEL_NAME,
            "stream": False,
            "format": ChunkExtraction.model_json_schema(),
            "options": {"temperature": 0, "num_ctx": 8192},
            "messages": [
                {"role": "system", "content": SYSTEM_PROMPT},
                {"role": "user", "content": user_prompt},
            ],
        },
        timeout=300,
    )
    response.raise_for_status()
    return ChunkExtraction.model_validate_json(response.json()["message"]["content"])

In [9]:
# check: run on ONE chunk and look at the raw result
sample_chunk = chunk_by_id["GPT-4_0011"] if "GPT-4_0011" in chunk_by_id else chunks[10]
print(sample_chunk["chunk_id"], "|", " > ".join(sample_chunk["section_path"]))
print(sample_chunk["text"][:600], "\n")

start = time.time()
sample_extraction = extract_chunk(sample_chunk)
print(f"took {time.time() - start:.1f}s\n")

for e in sample_extraction.entities:
    print(f"  ENTITY  {e.name:40} {e.entity_type:12} {e.aliases}")
for r in sample_extraction.relationships:
    print(f"  REL     {r.source} --{r.relation}--> {r.target}\n          \"{r.evidence}\"")

GPT-4_0011 | 3 Predictable Scaling > 3.1 Loss Prediction
The final loss of properly-trained large language models is thought to be well approximated by power laws in the amount of compute used to train the model [41, 42, 2, 14, 15]. To verify the scalability of our optimization infrastructure, we predicted GPT-4’s final loss on our internal codebase (not part of the training set) by fitting a scaling law with an irreducible loss term (as in Henighan et al. [15]): L(C) = aC + c, from models trained using the same methodology but using at most 10,000x less compute than GPT-4.

This prediction was made shortly after the run started, without use of any  

took 19.3s

  ENTITY  power law                                CONCEPT      []
  ENTITY  GPT-4                                    MODEL        []
  ENTITY  final loss                               METRIC       []
  ENTITY  scaling law                              METHOD       []
  ENTITY  Henighan et al.                          ORGANIZAT

## 5. Normalize names

The same thing gets written many ways (`LLMs`, `large language models`, `Large language model`).
`canonical()` turns every spelling into **one** name:
1. tidy up the text: spaces, `GPT--4` → `GPT-4`
2. look it up in `ALIASES` (lowercase keys)
3. otherwise keep the name as written. Matching is case-insensitive, so "Artificial intelligence" = "artificial intelligence".

`ALIASES` starts with a few known names. The model's `aliases` field adds more for each chunk.

In [10]:
ALIASES = {
    "llm": "large language model",
    "llms": "large language model",
    "large language models": "large language model",
    "language models": "language model",
    "lm": "language model",
    "agi": "artificial general intelligence",
    "ai": "artificial intelligence",
    "helm": "Holistic Evaluation of Language Models",
    "foundation models": "foundation model",
    "gpt4": "GPT-4",
    "gpt-4 model": "GPT-4",
    "rlhf": "reinforcement learning from human feedback",
    "cot": "chain-of-thought prompting",
}


def tidy(text):
    text = text.replace("“", '"').replace("”", '"').replace("’", "'").replace("‘", "'")
    text = text.replace("--", "-").replace("–", "-").replace("—", "-")
    return re.sub(r"\s+", " ", text).strip()


def canonical(name, extra_aliases=None):
    name = tidy(name)
    key = name.lower()
    if extra_aliases and key in extra_aliases:
        return extra_aliases[key]
    return ALIASES.get(key, name)


def key_of(name):
    """Case-insensitive key used to compare names."""
    return canonical(name).lower()

In [11]:
# check: different spellings should give the same key
tests = ["LLMs", "large language models", "Large language model", "GPT--4", "gpt-4",
         "HELM", "Holistic Evaluation of Language Models", "AGI", "Artificial General Intelligence"]
for t in tests:
    print(f"{t:42} -> {key_of(t)}")

LLMs                                       -> large language model
large language models                      -> large language model
Large language model                       -> large language model
GPT--4                                     -> gpt-4
gpt-4                                      -> gpt-4
HELM                                       -> holistic evaluation of language models
Holistic Evaluation of Language Models     -> holistic evaluation of language models
AGI                                        -> artificial general intelligence
Artificial General Intelligence            -> artificial general intelligence


## 6. Validate every relationship

A relationship is **accepted** only if all checks pass. Otherwise it's **rejected** with a reason, so you can see *why*.

| Check | Catches |
|---|---|
| entity name appears in the passage (otherwise the entity is dropped) | invented entities |
| source and target are extracted entities | made-up endpoints like "early writings" |
| source ≠ target | self-loops |
| entity is not a citation ("et al.", "[..]") | authors and references as entities |
| quote found in the passage (after tidying quotes, dashes and **removing citation markers** like `[362]`) | invented quotes |
| at least one name is **in the quote**, and **both** are in the quote's sentence | quotes that don't talk about the pair, while still allowing "GPT-4 … passing a simulated bar exam" |
| type rules (`DEVELOPED_BY` → ORGANIZATION, `EVALUATES` → MODEL, ORGANIZATION only in `DEVELOPED_BY`…) | "GPT-4 DEVELOPED_BY mathematics", "Restaurant PART_OF Mei et al." |
| one relation per entity pair per chunk | "GPT-4 IS_A LLM" + "GPT-4 PART_OF LLM" duplicates |

In [12]:
CITATION = re.compile(r"\[[^\]]{1,40}\]"                       # [362], [Ope23], [Devlin et al. 2019]
                      r"|\((?:[^()]*et al\.?[^()]*|[^()]*\b(?:19|20)\d\d[a-z]?)\)")   # (Konda et al., 2016), (2021)


def for_matching(text):
    """Text used for quote matching: tidy, no citation markers, lowercase."""
    text = CITATION.sub("", tidy(text))
    return re.sub(r"\s+", " ", text).strip().lower()


def is_citation(name):
    return "et al" in name.lower() or bool(re.search(r"\[|\]", name))


def sentence_around(passage, start, end):
    """The full sentence(s) that contain passage[start:end]."""
    left = max(passage.rfind(". ", 0, start), passage.rfind("? ", 0, start), passage.rfind("! ", 0, start))
    left = 0 if left == -1 else left + 2
    # search from the quote's last character, so a quote ending in "." stops at its own sentence
    ends = [i for i in (passage.find(". ", end - 1), passage.find("? ", end - 1), passage.find("! ", end - 1)) if i != -1]
    right = min(ends) + 1 if ends else len(passage)
    return passage[left:right]


def mentions(quote_key, entity_name, entity_aliases):
    """True if the entity's name, an alias, or a known ALIASES spelling appears in the quote
    as a whole word (an optional plural "s" is allowed, so "LLMs" matches "LLM")."""
    names = [entity_name] + entity_aliases
    names += [k for k, v in ALIASES.items() if v.lower() == entity_name.lower()]
    for n in names:
        pattern = r"(?<![a-z0-9])" + re.escape(tidy(n).lower()) + r"s?(?![a-z0-9])"
        if n and re.search(pattern, quote_key):
            return True
    return False


def validate(chunk, extraction):
    passage_key = for_matching(chunk["text"])

    # entity lookup: canonical key -> entity info (with aliases from this chunk)
    chunk_aliases = {}
    for e in extraction.entities:
        for a in e.aliases:
            chunk_aliases[tidy(a).lower()] = canonical(e.name)
    entities = {}
    for e in extraction.entities:
        if is_citation(e.name):                                # "Devlin et al." is not an entity
            continue
        if not mentions(tidy(chunk["text"]).lower(), e.name, e.aliases):   # entity must really be in the passage
            continue
        name = canonical(e.name, chunk_aliases)
        entities[name.lower()] = {"name": name, "type": e.entity_type,
                                  "aliases": [e.name] + e.aliases}

    seen_pairs = set()
    accepted, rejected = [], []
    for r in extraction.relationships:
        src = entities.get(canonical(r.source, chunk_aliases).lower())
        tgt = entities.get(canonical(r.target, chunk_aliases).lower())
        quote_key = for_matching(r.evidence)
        position = passage_key.find(quote_key) if quote_key else -1
        sentence = sentence_around(passage_key, position, position + len(quote_key)) if position >= 0 else ""

        if src is None:
            reason = "source not in entities"
        elif tgt is None:
            reason = "target not in entities"
        elif src["name"] == tgt["name"]:
            reason = "source == target"
        elif not quote_key:
            reason = "empty quote"
        elif position < 0:
            reason = "quote not in passage"
        elif not (mentions(quote_key, src["name"], src["aliases"]) or mentions(quote_key, tgt["name"], tgt["aliases"])):
            reason = "neither entity is in the quote"
        elif not mentions(sentence, src["name"], src["aliases"]):
            reason = "source not in the quote's sentence"
        elif not mentions(sentence, tgt["name"], tgt["aliases"]):
            reason = "target not in the quote's sentence"
        elif "ORGANIZATION" in (src["type"], tgt["type"]) and r.relation != "DEVELOPED_BY":
            reason = f"ORGANIZATION only allowed in DEVELOPED_BY, not {r.relation}"
        elif src["type"] not in TYPE_RULES.get((r.relation, "source"), {src["type"]}):
            reason = f"source type {src['type']} not allowed for {r.relation}"
        elif tgt["type"] not in TYPE_RULES.get((r.relation, "target"), {tgt["type"]}):
            reason = f"target type {tgt['type']} not allowed for {r.relation}"
        elif (src["name"], tgt["name"]) in seen_pairs:
            reason = "duplicate: this pair already has a relation in this chunk"
        else:
            reason = None
            seen_pairs.add((src["name"], tgt["name"]))

        row = {
            "source": src["name"] if src else r.source,
            "relation": r.relation,
            "target": tgt["name"] if tgt else r.target,
            "evidence": r.evidence,
            "chunk_id": chunk["chunk_id"],
            "document": chunk["document"],
            "section": " > ".join(chunk["section_path"]),
            "page": chunk["page_start"],
        }
        if reason:
            rejected.append({**row, "reason": reason})
        else:
            accepted.append(row)

    entity_rows = [{"name": v["name"], "type": v["type"], "chunk_id": chunk["chunk_id"],
                    "document": chunk["document"]} for v in entities.values()]
    return entity_rows, accepted, rejected

In [13]:
# check: validation on a hand-made example (no LLM needed)
fake_chunk = {"chunk_id": "demo", "document": "demo", "section_path": ["demo"], "page_start": 1,
              "text": "The latest model developed by OpenAI, GPT-4, beats ChatGPT. "
                      "GPT-4 can solve tasks that span mathematics, coding, vision, medicine and law. "
                      "We use the Beer dataset from Mei et al. (2021) [12]."}
fake = ChunkExtraction(
    entities=[Entity(name="GPT-4", entity_type="MODEL"), Entity(name="OpenAI", entity_type="ORGANIZATION"),
              Entity(name="ChatGPT", entity_type="MODEL"), Entity(name="mathematics", entity_type="DOMAIN"),
              Entity(name="coding", entity_type="DOMAIN"), Entity(name="vision", entity_type="DOMAIN"),
              Entity(name="medicine", entity_type="DOMAIN"), Entity(name="Beer", entity_type="DATASET"),
              Entity(name="Mei et al.", entity_type="ORGANIZATION")],
    relationships=[
        Relationship(source="GPT--4", relation="DEVELOPED_BY", target="OpenAI",
                     evidence="model developed by OpenAI, GPT--4"),                    # ok (dash fixed)
        Relationship(source="GPT-4", relation="OUTPERFORMS", target="ChatGPT",
                     evidence="The latest model developed by OpenAI"),               # neither name in quote
        Relationship(source="GPT-4", relation="DEVELOPED_BY", target="ChatGPT",
                     evidence="GPT-4, beats ChatGPT"),                               # wrong type
        Relationship(source="GPT-4", relation="HAS_CAPABILITY", target="law",
                     evidence="span mathematics, coding, vision, medicine and law"), # law not an entity
        Relationship(source="GPT-4", relation="HAS_CAPABILITY", target="mathematics",
                     evidence="span mathematics, coding"),                           # ok: GPT-4 is in the sentence
        Relationship(source="GPT-4", relation="IS_A", target="mathematics",
                     evidence="span mathematics, coding"),                           # duplicate pair
        Relationship(source="Beer", relation="PART_OF", target="Mei et al.",
                     evidence="the Beer dataset from Mei et al. (2021)"),           # citation entity dropped
        Relationship(source="GPT-4", relation="HAS_CAPABILITY", target="coding",
                     evidence="We use the Beer dataset from Mei et al. [12]"),     # quote found (citations ignored), but GPT-4 is not in that sentence
    ],
)
_, ok, bad = validate(fake_chunk, fake)
print("accepted:"); display(pd.DataFrame(ok)[["source", "relation", "target"]])
print("rejected:"); display(pd.DataFrame(bad)[["source", "relation", "target", "reason"]])

accepted:


,source,relation,target
0,GPT-4,DEVELOPED_BY,OpenAI
1,GPT-4,HAS_CAPABILITY,mathematics


rejected:


,source,relation,target,reason
0,GPT-4,OUTPERFORMS,ChatGPT,neither entity is in the quote
1,GPT-4,DEVELOPED_BY,ChatGPT,target type MODEL not allowed for DEVELOPED_BY
2,GPT-4,HAS_CAPABILITY,law,target not in entities
3,GPT-4,IS_A,mathematics,duplicate: this pair already has a relation in...
4,Beer,PART_OF,Mei et al.,target not in entities
5,GPT-4,HAS_CAPABILITY,coding,neither entity is in the quote


## 7. Pick the test set: 3 chunks from each paper

We skip sections with no useful relations (front matter, acknowledgments, author lists, prompt-example appendices).
Then we take chunks that mention our key entities (GPT-4, HELM, foundation model…), spread out over each paper.

In [14]:
SKIP_WORDS = ["front matter", "acknowledg", "author contributions", "rbrm instructions",
              "table full examples", "table examples", "prompt 1", "system card"]
KEYWORDS = ["gpt-4", "helm", "foundation model", "calibration", "openai", "benchmark", "outperform"]
PER_DOCUMENT = 3


def is_useful(chunk):
    section = " > ".join(chunk["section_path"]).lower()
    return not any(word in section for word in SKIP_WORDS)


test_chunks = []
for document in sorted({c["document"] for c in chunks}):
    candidates = [c for c in chunks
                  if c["document"] == document and is_useful(c)
                  and any(k in c["text"].lower() for k in KEYWORDS)]
    step = max(1, len(candidates) // PER_DOCUMENT)
    test_chunks += candidates[::step][:PER_DOCUMENT]

In [15]:
# check: which chunks are in the test set?
display(pd.DataFrame([{
    "chunk_id": c["chunk_id"],
    "section": " > ".join(c["section_path"])[:60],
    "keywords": [k for k in KEYWORDS if k in c["text"].lower()],
} for c in test_chunks]))

,chunk_id,section,keywords
0,AGI_0001,Abstract,"[gpt-4, openai]"
1,AGI_0157,4.3 Mathematical modeling in various domains,[gpt-4]
2,AGI_0325,9.2 Misinformation and manipulation,[gpt-4]
3,Foundation_Models_0004,1 INTRODUCTION,[foundation model]
4,Foundation_Models_0225,3 APPLICATIONS > 3.1 Healthcare and biomedicine,[foundation model]
5,Foundation_Models_0465,4 TECHNOLOGY > 4.7 Security and privacy > 4.7....,[foundation model]
6,GPT-4_0001,Abstract,"[gpt-4, benchmark]"
7,GPT-4_0072,A Exam Benchmark Methodology > A.4 Images,[gpt-4]
8,GPT-4_0164,2 GPT-4 Observed Safety Challenges > 2.3 Harmf...,[gpt-4]
9,Holistic_Eval_0001,Abstract,[helm]


## 8. Run extraction with checkpointing

Each chunk's result is **appended to a `.jsonl` file as soon as it's done**. If the run stops (crash, closed laptop, Ollama error), just run the cell again: it **skips chunks already in the file** and **retries chunks that errored**.
This same function will process all ~2,900 chunks later; only the list and the file name change.

In [16]:
def load_done(path):
    if not path.exists():
        return {}
    with path.open(encoding="utf-8") as f:
        rows = [json.loads(line) for line in f if line.strip()]
    return {row["chunk_id"]: row for row in rows}


def run_extraction(chunk_list, output_path):
    done = load_done(output_path)
    # skip chunks that are done; retry the ones that errored last time
    todo = [c for c in chunk_list if c["chunk_id"] not in done or done[c["chunk_id"]]["error"]]
    print(f"{len(chunk_list) - len(todo)} already done, {len(todo)} to go")

    for i, chunk in enumerate(todo, start=1):
        start = time.time()
        try:
            extraction = extract_chunk(chunk)
            entity_rows, accepted, rejected = validate(chunk, extraction)
            row = {"chunk_id": chunk["chunk_id"], "entities": entity_rows,
                   "accepted": accepted, "rejected": rejected, "error": None}
        except Exception as exc:
            row = {"chunk_id": chunk["chunk_id"], "entities": [],
                   "accepted": [], "rejected": [], "error": str(exc)}

        with output_path.open("a", encoding="utf-8") as f:     # checkpoint: save right away
            f.write(json.dumps(row, ensure_ascii=False) + "\n")

        print(f"[{i}/{len(todo)}] {chunk['chunk_id']:25} "
              f"entities={len(row['entities']):2} accepted={len(row['accepted']):2} "
              f"rejected={len(row['rejected']):2} {time.time() - start:5.1f}s"
              + (f"  ERROR: {row['error'][:80]}" if row["error"] else ""))

    return load_done(output_path)

In [17]:
TEST_OUTPUT = ARTIFACTS_DIR / "extractions_test_v3.jsonl"
test_results = run_extraction(test_chunks, TEST_OUTPUT)

15 already done, 0 to go


## 9. Review the results

Three checks, matching the three questions:
1. **Entity quality**: types and names, and whether any name got two different types.
2. **Relationship quality**: accepted relations with their quotes. Read them!
3. **Evidence quality**: why things were rejected.

In [18]:
results = [test_results[c["chunk_id"]] for c in test_chunks if c["chunk_id"] in test_results]
all_entities = [e for r in results for e in r["entities"]]
all_accepted = [a for r in results for a in r["accepted"]]
all_rejected = [x for r in results for x in r["rejected"]]

print("chunks:", len(results), "| errors:", sum(r["error"] is not None for r in results))
print("entities:", len(all_entities), "| accepted relations:", len(all_accepted),
      "| rejected relations:", len(all_rejected))
print("\naccepted by relation type:", dict(Counter(a["relation"] for a in all_accepted)))
print("rejected by reason:", dict(Counter(x["reason"] for x in all_rejected)))

chunks: 15 | errors: 0
entities: 77 | accepted relations: 17 | rejected relations: 35

accepted by relation type: {'DEVELOPED_BY': 1, 'IS_A': 5, 'HAS_RISK': 2, 'HAS_CAPABILITY': 3, 'EVALUATED_ON': 1, 'HAS_LIMITATION': 1, 'PART_OF': 3, 'BUILDS_ON': 1}
rejected by reason: {'target type CONCEPT not allowed for HAS_CAPABILITY': 4, 'neither entity is in the quote': 3, "source not in the quote's sentence": 8, 'target not in entities': 9, "target not in the quote's sentence": 1, 'source type CONCEPT not allowed for HAS_CAPABILITY': 1, 'source not in entities': 3, 'source type BENCHMARK not allowed for HAS_LIMITATION': 1, 'target type CONCEPT not allowed for EVALUATES': 1, 'target type DOMAIN not allowed for EVALUATES': 4}


In [19]:
# check 1: entity quality
entities_df = pd.DataFrame(all_entities, columns=["name", "type", "chunk_id", "document"])
display(entities_df.groupby(["type"]).size().rename("count").to_frame().T)

summary = entities_df.groupby("name").agg(types=("type", lambda t: sorted(set(t))),
                                          mentions=("type", "size"))
print("names with MORE THAN ONE type (should be fixed or added to ALIASES):")
display(summary[summary["types"].map(len) > 1])
print("most frequent entities:")
display(summary.sort_values("mentions", ascending=False).head(15))

type,BENCHMARK,CONCEPT,DATASET,DOMAIN,METHOD,METRIC,MODEL,ORGANIZATION
count,12,32,6,12,1,2,11,1


names with MORE THAN ONE type (should be fixed or added to ALIASES):


,types,mentions
name,,


most frequent entities:


,types,mentions
name,,
GPT-4,[MODEL],4
artificial intelligence,[CONCEPT],2
Holistic Evaluation of Language Models,[BENCHMARK],2
CLIP,[MODEL],2
large language model,[CONCEPT],2
foundation model,[CONCEPT],2
BERT,[MODEL],1
AP Psychology,[BENCHMARK],1
AP English Literature,[BENCHMARK],1


In [20]:
# check 2: relationship quality - read every accepted relation
pd.set_option("display.max_colwidth", 120)
display(pd.DataFrame(all_accepted, columns=["source", "relation", "target", "evidence", "chunk_id"]))

,source,relation,target,evidence,chunk_id
0,GPT-4,DEVELOPED_BY,OpenAI,"The latest model developed by OpenAI, GPT-4 [Ope23], was trained using an unprecedented scale of compute and data.",AGI_0001
1,GPT-4,IS_A,large language model,GPT-4 is part of a new cohort of LLMs (along with ChatGPT and Google’s PaLM for example),AGI_0001
2,GPT-4,HAS_RISK,cyberattacks,The powers of generalization and interaction of models like GPT-4 can be harnessed to increase the scope and magnitu...,AGI_0325
3,GPT-4,HAS_RISK,persuasion,GPT-4 and descendants can be harnessed to contextualize and personalize interactions to maximize the impact of their...,AGI_0325
4,BERT,IS_A,foundation model,"current examples include BERT [Devlin et al. 2019], GPT-3 [Brown et al. 2020], and CLIP [Radford et al. 2021]",Foundation_Models_0004
5,GPT-3,IS_A,foundation model,"current examples include BERT [Devlin et al. 2019], GPT-3 [Brown et al. 2020], and CLIP [Radford et al. 2021]",Foundation_Models_0004
6,CLIP,IS_A,foundation model,"current examples include BERT [Devlin et al. 2019], GPT-3 [Brown et al. 2020], and CLIP [Radford et al. 2021]",Foundation_Models_0004
7,GPT-3,HAS_CAPABILITY,natural language,GPT-3 has 175 billion parameters and can be adapted via natural language prompts to do a passable job on a wide rang...,Foundation_Models_0004
8,GPT-4,IS_A,multimodal model,"GPT-4, a large-scale, multimodal model which can accept image and text inputs and produce text outputs",GPT-4_0001
9,GPT-4,HAS_CAPABILITY,professional,exhibits human-level performance on various professional and academic benchmarks,GPT-4_0001


In [21]:
# check 3: evidence quality - what got rejected, and why
display(pd.DataFrame(all_rejected, columns=["source", "relation", "target", "reason", "evidence"]))

,source,relation,target,reason,evidence
0,GPT-4,HAS_CAPABILITY,general intelligence,target type CONCEPT not allowed for HAS_CAPABILITY,GPT-4 is part of a new cohort of LLMs (along with ChatGPT and Google’s PaLM for example) that exhibit more general i...
1,GPT-4,HAS_RISK,disinformation,neither entity is in the quote,"Like any powerful technology, LLMs can be used by malevolent actors to do damage."
2,GPT-4,HAS_RISK,manipulation,neither entity is in the quote,"The interactive powers and models of human judgment and decision making can be employed to manipulate, persuade, or ..."
3,GPT-4,HAS_RISK,disinformation,source not in the quote's sentence,"new powers of efficiency and scale can be enabled with automation using the LLMs, including uses aimed at constructi..."
4,foundation model,IS_A,model,target not in entities,A foundation model is any model that is trained on broad data (generally using self-supervision at scale) that can b...
5,foundation model,USES,self-supervised learning,target not in the quote's sentence,a foundation model is any model that is trained on broad data (generally using self-supervision at scale)
6,foundation model,HAS_CAPABILITY,natural language,source type CONCEPT not allowed for HAS_CAPABILITY,can be adapted via natural language prompts to do a passable job on a wide range of tasks
7,foundation model,HAS_CAPABILITY,healthcare,source not in the quote's sentence,Healthcare and biomedicine are an enormous application area in society
8,foundation model,HAS_CAPABILITY,biomedicine,source not in the quote's sentence,Healthcare and biomedicine are an enormous application area in society
9,CLIP,HAS_CAPABILITY,distributional shifts,target type CONCEPT not allowed for HAS_CAPABILITY,multimodal models such as CLIP are surprisingly robust to (non-adversarial) distributional shifts


## 10. Grade it by hand

The checks can't tell whether a relation is *true*, only whether it's well-formed. So:
1. Run the cell below. It writes `artifacts/review_test_v3.csv` with an empty `verdict` column.
2. Open it in Excel and fill `verdict` with `correct`, `partial` or `wrong` for each row.
3. Run the next cell to get the precision.

Aim for **≥ 80% correct** before running all chunks. If it's lower, look at the wrong rows: they tell you which prompt example, relation definition or alias to change.

In [22]:
REVIEW_PATH = ARTIFACTS_DIR / "review_test_v3.csv"

review = pd.DataFrame(all_accepted, columns=["chunk_id", "source", "relation", "target", "evidence"])
review["verdict"] = ""
if not REVIEW_PATH.exists():               # never overwrite your grading
    review.to_csv(REVIEW_PATH, index=False, encoding="utf-8-sig")
    print("written:", REVIEW_PATH)
else:
    print("already exists (not overwritten):", REVIEW_PATH)

already exists (not overwritten): c:\Users\Swapn\OneDrive\Documents\GitHub\OKF\artifacts\review_test_v3.csv


In [23]:
# check: precision from your grading
graded = pd.read_csv(REVIEW_PATH)
graded["verdict"] = graded["verdict"].fillna("").str.strip().str.lower()
done = graded[graded["verdict"] != ""]

print(f"graded {len(done)} of {len(graded)}")
if len(done):
    counts = done["verdict"].value_counts()
    print(counts.to_string())
    print(f"\nprecision (correct only): {counts.get('correct', 0) / len(done):.0%}")
    print(f"precision (correct + partial): {(counts.get('correct', 0) + counts.get('partial', 0)) / len(done):.0%}")
    display(done[done["verdict"] == "wrong"][["source", "relation", "target", "evidence"]])

graded 17 of 17
verdict
correct    12
partial     4
wrong       1

precision (correct only): 71%
precision (correct + partial): 94%


,source,relation,target,evidence
7,GPT-3,HAS_CAPABILITY,natural language,GPT-3 has 175 billion parameters and can be adapted via natural language prompts to do a passable job on a wide rang...


# Part 2: Finalize graph facts

**confirm review → full extraction → load facts → normalize entity identities → final facts file**

The output is two files the graph is built from:
- `artifacts/graph_entities.csv`: one row per real-world thing (GPT-4 appears once, not 40 times)
- `artifacts/graph_facts.jsonl`: one row per (source, relation, target), with **all** the quotes and chunks that support it

## 11. Confirm the review results

A **quality gate**: we only spend ~8 hours on the full run if the graded test set is good enough.
It reads your graded `review_test_v3.csv`, the last test you graded.

In [24]:
GATE_REVIEW = ARTIFACTS_DIR / "review_test_v3.csv"
MIN_STRICT = 0.60           # share of "correct"
MIN_LENIENT = 0.80          # share of "correct" + "partial"

graded = pd.read_csv(GATE_REVIEW)
graded["verdict"] = graded["verdict"].fillna("").str.strip().str.lower()
graded = graded[graded["verdict"] != ""]

strict = (graded["verdict"] == "correct").mean()
lenient = graded["verdict"].isin(["correct", "partial"]).mean()

print(f"graded relations: {len(graded)}")
print(f"strict precision : {strict:.0%}  (need {MIN_STRICT:.0%})")
print(f"lenient precision: {lenient:.0%}  (need {MIN_LENIENT:.0%})")

GATE_PASSED = len(graded) >= 10 and strict >= MIN_STRICT and lenient >= MIN_LENIENT
print("\nGATE PASSED - go ahead with the full run" if GATE_PASSED
      else "\nGATE FAILED - improve the prompt/rules and re-test before the full run")

graded relations: 17
strict precision : 71%  (need 60%)
lenient precision: 94%  (need 80%)

GATE PASSED - go ahead with the full run


## 12. Full extraction (all useful chunks)

Same `run_extraction()` as the test, just on every useful chunk and into `extractions_all.jsonl`.
- **Stop any time** (interrupt the kernel). Running the cell again continues where it stopped.
- Run the **status cell** first to see what's left. It doesn't call the LLM.
- Expect ~10 s per chunk, so the whole set takes several hours. Good to leave running overnight.

In [25]:
FULL_OUTPUT = ARTIFACTS_DIR / "extractions_all.jsonl"
useful_chunks = [c for c in chunks if is_useful(c)]

# status (no LLM calls)
done = load_done(FULL_OUTPUT)
finished = [cid for cid, r in done.items() if not r["error"]]
remaining = len(useful_chunks) - len(finished)
print(f"useful chunks: {len(useful_chunks)} of {len(chunks)}")
print(f"done: {len(finished)} | errors to retry: {sum(1 for r in done.values() if r['error'])} | remaining: {remaining}")
print(f"estimated time left at ~10 s/chunk: {remaining * 10 / 3600:.1f} hours")

useful chunks: 2835 of 2927
done: 0 | errors to retry: 0 | remaining: 2835
estimated time left at ~10 s/chunk: 7.9 hours


In [ ]:
# the long run - safe to stop and re-run
assert GATE_PASSED, "Quality gate failed (section 11). Fix extraction first."
all_results = run_extraction(useful_chunks, FULL_OUTPUT)

0 already done, 2835 to go
[1/2835] AGI_0001                  entities= 7 accepted= 3 rejected= 1  11.1s
[2/2835] AGI_0002                  entities= 9 accepted= 8 rejected= 0  15.4s
[3/2835] AGI_0003                  entities= 5 accepted= 0 rejected= 6  13.2s
[4/2835] AGI_0004                  entities= 7 accepted= 0 rejected= 6  13.5s
[5/2835] AGI_0005                  entities=17 accepted= 8 rejected= 9  37.5s
[6/2835] AGI_0006                  entities=13 accepted= 4 rejected= 4  19.8s
[7/2835] AGI_0007                  entities= 3 accepted= 1 rejected= 1   5.9s
[8/2835] AGI_0008                  entities= 3 accepted= 1 rejected= 2   7.2s
[9/2835] AGI_0009                  entities= 0 accepted= 0 rejected= 0   2.6s
[10/2835] AGI_0010                  entities= 0 accepted= 0 rejected= 0   2.5s
[11/2835] AGI_0011                  entities= 6 accepted= 3 rejected= 3  13.9s
[12/2835] AGI_0012                  entities= 0 accepted= 0 rejected= 0   2.6s
[13/2835] AGI_0013                

In [ ]:
# check: is the full run complete?
all_results = load_done(FULL_OUTPUT)
missing = [c["chunk_id"] for c in useful_chunks if c["chunk_id"] not in all_results]
errors = [cid for cid, r in all_results.items() if r["error"]]
print(f"results: {len(all_results)} | missing: {len(missing)} | errors: {len(errors)}")
if missing or errors:
    print("NOT complete yet - re-run the extraction cell above")
else:
    print("Full extraction complete")

## 13. Load all accepted facts

Only **accepted** relations become graph facts. Rejected ones stay in the file for debugging.

In [ ]:
raw_facts = [fact for r in all_results.values() for fact in r["accepted"]]
raw_entities = [e for r in all_results.values() for e in r["entities"]]

facts_df = pd.DataFrame(raw_facts)
entities_df = pd.DataFrame(raw_entities)
print(f"accepted facts: {len(facts_df)} | entity mentions: {len(entities_df)}")

In [ ]:
# check: where do facts come from, and what kinds are they?
display(facts_df.groupby("document").size().rename("facts").to_frame().T)
display(facts_df["relation"].value_counts().rename("count").to_frame().T)
print("chunks that produced at least one fact:", facts_df["chunk_id"].nunique(), "of", len(all_results))

## 14. Normalize entity identities across papers

Each chunk was normalized on its own. Across 2,900 chunks the same thing still shows up in different spellings:
`GPT-4` / `GPT 4` / `gpt-4`, `large language model` / `large language models`, `chain-of-thought` / `chain of thought`.

We give every name an **identity key** and merge names with the same key:
1. `canonical()` + the `ALIASES` map (from section 5)
2. lowercase, and remove spaces, hyphens and dots, so `GPT-4` = `gpt 4` = `GPT4`
3. a plural is merged **only if the singular also exists** (`foundation models` → `foundation model`, but `bias` stays `bias`)
4. `MANUAL_MERGES` for anything the rules miss. Look at the check cells and add pairs there.

Each merged entity gets:
- **name**: its most common spelling
- **type**: its most common type
- **aliases**: all other spellings

In [ ]:
# add pairs you spot in the checks below:  "spelling to replace": "name to keep"
MANUAL_MERGES = {
    "gpt-4 (launch)": "GPT-4",
    "gpt-4 (early)": "GPT-4",
    "openai gpt-4": "GPT-4",
}


def squash(name):
    """Identity key: canonical, lowercase, no spaces/hyphens/dots/underscores."""
    name = MANUAL_MERGES.get(tidy(name).lower(), name)
    return re.sub(r"[\s\-_.]", "", canonical(name).lower())


# 1) key for every spelling
spellings = Counter(entities_df["name"].tolist() + facts_df["source"].tolist() + facts_df["target"].tolist())
key_of_name = {name: squash(name) for name in spellings}

# 2) plural -> singular, only when the singular exists
all_keys = set(key_of_name.values())
for name, key in key_of_name.items():
    if key.endswith("s") and not key.endswith("ss") and key[:-1] in all_keys:
        key_of_name[name] = key[:-1]

# 3) one display name, type and alias list per key
groups = {}
for name, count in spellings.items():
    groups.setdefault(key_of_name[name], Counter())[name] += count

type_votes = {}
for row in entities_df.itertuples():
    type_votes.setdefault(key_of_name[row.name], Counter())[row.type] += 1

entity_rows = []
for key, names in groups.items():
    display_name = canonical(names.most_common(1)[0][0])         # e.g. "HELM" -> full name from ALIASES
    for preferred in names:                                   # prefer the ALIASES / MANUAL_MERGES target spelling
        if preferred in set(ALIASES.values()) | set(MANUAL_MERGES.values()):
            display_name = preferred
    votes = type_votes.get(key, Counter({"CONCEPT": 1}))
    entity_rows.append({
        "key": key,
        "name": display_name,
        "type": votes.most_common(1)[0][0],
        "type_votes": dict(votes),
        "aliases": sorted(n for n in names if n != display_name),
        "mentions": sum(names.values()),
    })

entities_final = pd.DataFrame(entity_rows).set_index("key")
name_of_key = entities_final["name"].to_dict()
final_name = {name: name_of_key[key] for name, key in key_of_name.items()}

print(f"{len(spellings)} spellings -> {len(entities_final)} entities")

In [ ]:
# check: which spellings were merged? (read these - wrong merges are easy to spot here)
merged = entities_final[entities_final["aliases"].map(len) > 0].sort_values("mentions", ascending=False)
print(f"{len(merged)} entities have more than one spelling")
display(merged[["name", "type", "aliases", "mentions"]].head(30))

In [ ]:
# check: entities that got DIFFERENT types in different chunks (majority vote decided)
conflicts = entities_final[entities_final["type_votes"].map(len) > 1]
print(f"{len(conflicts)} entities with conflicting types")
display(conflicts[["name", "type", "type_votes", "mentions"]].sort_values("mentions", ascending=False).head(20))

In [ ]:
# check: near-duplicates the rules did NOT merge (same first word + same type) - candidates for MANUAL_MERGES
candidates = (entities_final.assign(first=entities_final["name"].str.lower().str.split().str[0])
              .groupby(["first", "type"])["name"].apply(list))
candidates = candidates[candidates.map(len) > 1]
display(candidates[candidates.index.get_level_values("type").isin(["MODEL", "BENCHMARK", "ORGANIZATION"])].head(20))

## 15. Final graph facts

Rename every fact's source and target to the merged entity names, then:
- **drop self-loops** that merging can create (`GPT 4 → GPT-4`)
- **merge duplicate facts**: the same `(source, relation, target)` found in many chunks becomes one fact, with a `support` count and all its quotes and chunk IDs. Facts found in several chunks or papers are the most trustworthy.
- keep only entities that appear in **at least one fact**. Loose entities ("image", "prompt") never enter the graph.

In [ ]:
facts_df["source_final"] = facts_df["source"].map(final_name)
facts_df["target_final"] = facts_df["target"].map(final_name)
facts_df = facts_df[facts_df["source_final"] != facts_df["target_final"]]          # no self-loops

graph_facts = (
    facts_df.groupby(["source_final", "relation", "target_final"])
    .agg(support=("chunk_id", "nunique"),
         documents=("document", lambda d: sorted(set(d))),
         chunk_ids=("chunk_id", lambda c: sorted(set(c))),
         evidence=("evidence", lambda e: list(dict.fromkeys(e))[:5]),     # up to 5 different quotes
         pages=("page", lambda p: sorted(set(p))))
    .reset_index()
    .rename(columns={"source_final": "source", "target_final": "target"})
    .sort_values("support", ascending=False)
)

used = set(graph_facts["source"]) | set(graph_facts["target"])
graph_entities = entities_final[entities_final["name"].isin(used)].reset_index()

print(f"facts: {len(facts_df)} raw -> {len(graph_facts)} unique")
print(f"entities: {len(entities_final)} total -> {len(graph_entities)} used in facts")

In [ ]:
# check: the most supported facts, and how much the papers connect to each other
pd.set_option("display.max_colwidth", 80)
display(graph_facts.head(15)[["source", "relation", "target", "support", "documents"]])

docs_per_entity = (pd.concat([graph_facts[["source", "documents"]].rename(columns={"source": "name"}),
                              graph_facts[["target", "documents"]].rename(columns={"target": "name"})])
                   .explode("documents").groupby("name")["documents"].nunique())
cross = docs_per_entity[docs_per_entity >= 2].sort_values(ascending=False)
print(f"\nentities appearing in 2+ papers (these link the papers together): {len(cross)}")
print(cross.head(15).to_string())

In [ ]:
# save the final graph facts + entities
FACTS_PATH = ARTIFACTS_DIR / "graph_facts.jsonl"
ENTITIES_PATH = ARTIFACTS_DIR / "graph_entities.csv"

with FACTS_PATH.open("w", encoding="utf-8") as f:
    for row in graph_facts.to_dict("records"):
        f.write(json.dumps(row, ensure_ascii=False) + "\n")
graph_entities.to_csv(ENTITIES_PATH, index=False, encoding="utf-8-sig")

print("saved:", FACTS_PATH.name, f"({len(graph_facts)} facts)")
print("saved:", ENTITIES_PATH.name, f"({len(graph_entities)} entities)")

In [ ]:
# check: reload and verify every fact points to a known entity
reloaded = [json.loads(line) for line in FACTS_PATH.open(encoding="utf-8")]
known = set(pd.read_csv(ENTITIES_PATH)["name"])
bad = [f for f in reloaded if f["source"] not in known or f["target"] not in known]
print(f"facts in file: {len(reloaded)} | facts with unknown entities: {len(bad)}")
assert not bad, "some facts point to entities that are missing from graph_entities.csv"

### Next
Build the NetworkX graph from `graph_facts.jsonl` + `graph_entities.csv`, then do retrieval:
query → matching entities → 1–2 hop neighbours → their facts + source chunks.